# Calibration statistics for the curling Q-network

This demo loads the existing saved Q-network, creates a final-score dataset using the same sheet-state/throw setup as `training.ipynb`, computes the new statistics, and plots its categorical calibration.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Make imports work whether the notebook is launched from the repository root
# or from scratch/.
repo_root = Path.cwd()
if not (repo_root / 'curling_nn.py').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import bot
import curling_nn
import data_generation
import dataset
import physics
import scoring
import state
import stats
from constants import q_network_weights_path

%load_ext autoreload
%autoreload 2

In [ ]:
# Load the existing model and its feature normalizer.
neural_network, normalizer = curling_nn.load_q_weights(q_network_weights_path)
print(f'network outputs: {2 * neural_network.num_stones_per_side + 1}')
print(f'network input features: {normalizer.feature_means.size}')

## Build an evaluation dataset

This follows the dataset construction in `training.ipynb`: random sheet states, random throws, plus grid-search throws sampled by their realized score. The saved model's normalizer is applied to these raw features.

In [ ]:
rng = np.random.default_rng(2026)
np.random.seed(2026)
num_stones_per_side = 5
seed_states = data_generation.random_sheet_states(
    team1=num_stones_per_side,
    team2=num_stones_per_side - 1,
    num_sims=300,
)
team = 1

random_throws, random_states = bot.RandomThrows(
    rng=rng, n_throws_to_generate=1
).get_throws_for_num_sims(team=team, sheet_states=seed_states)
scored_throws, scored_states = data_generation.sample_throws_by_score_for_sheets(
    sheet_states=seed_states,
    team=team,
    throw_searcher=bot.ThrowsGridSearcher(10, 10, 4),
    n_per_score=5,
    rng=rng,
)
throws, states = data_generation.combine_throw_datasets(
    (random_throws, random_states),
    (scored_throws, scored_states),
)
final_states = physics.run_until_stopping(
    sheet_states=state.add_stones_from_throws(states, throws)
)
final_scores = scoring.get_net_score_for_team(final_states, 0)

# The helper creates the one-hot answers and raw feature matrix. Reuse its
# answers, but normalize with the saved model's normalizer.
raw_data = curling_nn.QInputFeatures.create_score_match_dataset_from_sheet_states(
    states, throws, final_scores, num_stones_per_side
)
evaluation_data = dataset.TrainingData(
    input_features=normalizer.normalize(raw_data.raw_inputs),
    answers=raw_data.answers,
    normalizer=normalizer,
    raw_inputs=raw_data.raw_inputs,
)
print(f'evaluation examples: {evaluation_data.size()}')

In [ ]:
model_stats = stats.compute_stats(
    neural_network,
    evaluation_data,
    score_values=np.arange(-num_stones_per_side, num_stones_per_side + 1),
    seed=2026,
)

print(f'expected-score R²: {model_stats.r_squared.value:.3f} ± {model_stats.r_squared.stderr:.3f}')
print(f'P(actual score): {model_stats.correct_score_probability.value:.3f} ± {model_stats.correct_score_probability.stderr:.3f}')
print(f'negative log P(actual): {model_stats.negative_log_probability.value:.3f} ± {model_stats.negative_log_probability.stderr:.3f}')

In [ ]:
calibration = model_stats.calibration
midpoints = np.array([(bucket.lower_bound + bucket.upper_bound) / 2 for bucket in calibration])
predicted = np.array([bucket.predicted_fraction for bucket in calibration])
observed = np.array([bucket.actual_fraction for bucket in calibration])
stderr = np.array([bucket.predicted_stderr for bucket in calibration])
counts = np.array([bucket.count for bucket in calibration])

fig, ax = plt.subplots(figsize=(7, 6))
nonempty = counts > 0
ax.errorbar(midpoints[nonempty], observed[nonempty], yerr=stderr[nonempty], fmt='o', capsize=3, label='observed fraction')
ax.plot([0, 1], [0, 1], '--', color='0.5', label='perfect calibration')
for x, y, n in zip(midpoints[nonempty], observed[nonempty], counts[nonempty]):
    ax.annotate(str(n), (x, y), xytext=(4, 4), textcoords='offset points', fontsize=8)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='predicted probability bucket', ylabel='observed event fraction', title='Final-score calibration')
ax.grid(alpha=0.25)
ax.legend()
plt.show()